## Bronze to Silver: Data Cleaning and Transformation for Dimension Tables

In [0]:
import pyspark.sql.functions as f 
from pyspark.sql.types import StringType, IntegerType, FloatType, StructType, DataType, TimestampType

In [0]:
catalog_name ='ecommerce'

## Brands

In [0]:
df_bronze = spark.table(f"{catalog_name}.bronze.brz_brands") 
# spark.table() reads an existing table from Spark and returns it as a DataFrame.
df_bronze.show(10)

+----------+-----------+-------------+--------------------+--------------------+
|brand_code| brand_name|category_code|        _source_file|         ingested_at|
+----------+-----------+-------------+--------------------+--------------------+
|      ACME|   AcmeTech|           CE|dbfs:/Volumes/eco...|2026-05-04 03:38:...|
|      NOVW|  NovaWave |           CE|dbfs:/Volumes/eco...|2026-05-04 03:38:...|
|      ZNTH|     Zenith|           CE|dbfs:/Volumes/eco...|2026-05-04 03:38:...|
|      BYTM|    ByteMax|           CE|dbfs:/Volumes/eco...|2026-05-04 03:38:...|
|      ECOT|    EcoTone|           CE|dbfs:/Volumes/eco...|2026-05-04 03:38:...|
|      SKYL|    SkyLink|           CE|dbfs:/Volumes/eco...|2026-05-04 03:38:...|
|     VOLT@|   VoltEdge|           CE|dbfs:/Volumes/eco...|2026-05-04 03:38:...|
|      PHTX|   Photonix|           CE|dbfs:/Volumes/eco...|2026-05-04 03:38:...|
|      URTL| UrbanTrail|          APP|dbfs:/Volumes/eco...|2026-05-04 03:38:...|
|      COTC| CottonClub|    

In [0]:
df_silver = df_bronze.withColumn("brand_name", f.trim(f.col("brand_name")))
df_silver.show(10)

+----------+----------+-------------+--------------------+--------------------+
|brand_code|brand_name|category_code|        _source_file|         ingested_at|
+----------+----------+-------------+--------------------+--------------------+
|      ACME|  AcmeTech|           CE|dbfs:/Volumes/eco...|2026-05-04 03:38:...|
|      NOVW|  NovaWave|           CE|dbfs:/Volumes/eco...|2026-05-04 03:38:...|
|      ZNTH|    Zenith|           CE|dbfs:/Volumes/eco...|2026-05-04 03:38:...|
|      BYTM|   ByteMax|           CE|dbfs:/Volumes/eco...|2026-05-04 03:38:...|
|      ECOT|   EcoTone|           CE|dbfs:/Volumes/eco...|2026-05-04 03:38:...|
|      SKYL|   SkyLink|           CE|dbfs:/Volumes/eco...|2026-05-04 03:38:...|
|     VOLT@|  VoltEdge|           CE|dbfs:/Volumes/eco...|2026-05-04 03:38:...|
|      PHTX|  Photonix|           CE|dbfs:/Volumes/eco...|2026-05-04 03:38:...|
|      URTL|UrbanTrail|          APP|dbfs:/Volumes/eco...|2026-05-04 03:38:...|
|      COTC|CottonClub|          APP|dbf

In [0]:
df_silver = df_silver.withColumn("brand_code", f.regexp_replace(f.col("brand_code"), r"[^a-zA-Z0-9]", ""))
df_silver.show(10)
# F.regexp_replace(column, pattern, replacement)
# replaces text matching a regex pattern, 
# here pattern is "[^a-zA-Z0-9]", which means "not a letter or a number"
# replacement is "", which means "replace with nothing"


+----------+----------+-------------+--------------------+--------------------+
|brand_code|brand_name|category_code|        _source_file|         ingested_at|
+----------+----------+-------------+--------------------+--------------------+
|      ACME|  AcmeTech|           CE|dbfs:/Volumes/eco...|2026-05-04 03:38:...|
|      NOVW|  NovaWave|           CE|dbfs:/Volumes/eco...|2026-05-04 03:38:...|
|      ZNTH|    Zenith|           CE|dbfs:/Volumes/eco...|2026-05-04 03:38:...|
|      BYTM|   ByteMax|           CE|dbfs:/Volumes/eco...|2026-05-04 03:38:...|
|      ECOT|   EcoTone|           CE|dbfs:/Volumes/eco...|2026-05-04 03:38:...|
|      SKYL|   SkyLink|           CE|dbfs:/Volumes/eco...|2026-05-04 03:38:...|
|      VOLT|  VoltEdge|           CE|dbfs:/Volumes/eco...|2026-05-04 03:38:...|
|      PHTX|  Photonix|           CE|dbfs:/Volumes/eco...|2026-05-04 03:38:...|
|      URTL|UrbanTrail|          APP|dbfs:/Volumes/eco...|2026-05-04 03:38:...|
|      COTC|CottonClub|          APP|dbf

In [0]:
df_silver.select("category_code").distinct().show()

+-------------+
|category_code|
+-------------+
|           CE|
|          APP|
|          HNK|
|          BPC|
|        BOOKS|
|          BKS|
|      GROCERY|
|         GRCY|
|          TOY|
|         TOYS|
|          SPT|
+-------------+



In [0]:
anomalies = {
    "GROCERY": "GRCY",
    "BOOKS": "BKS",
    "TOYS": "TOY"
}

# PySpark replace is easy
df_silver = df_silver.replace(to_replace=anomalies, subset=["category_code"])
df_silver.select('category_code').distinct().show()

+-------------+
|category_code|
+-------------+
|           CE|
|          APP|
|          HNK|
|          BPC|
|          BKS|
|         GRCY|
|          TOY|
|          SPT|
+-------------+



In [0]:
# Write raw data to the silver layer (catalog: ecommerce, schema: silver, table: slv_brands)
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_brands")

## Category

In [0]:
df_bronze = spark.table(f"{catalog_name}.bronze.brz_category")

df_bronze.show(10)

+-------------+--------------------+--------------------+--------------------+
|category_code|       category_name|        _ingested_at|        _source_file|
+-------------+--------------------+--------------------+--------------------+
|           ce|         Electronics|2026-05-12 12:30:...|dbfs:/Volumes/eco...|
|          app|             Apparel|2026-05-12 12:30:...|dbfs:/Volumes/eco...|
|          hnk|      Home & Kitchen|2026-05-12 12:30:...|dbfs:/Volumes/eco...|
|          bpc|Beauty & Personal...|2026-05-12 12:30:...|dbfs:/Volumes/eco...|
|          bks|               Books|2026-05-12 12:30:...|dbfs:/Volumes/eco...|
|         grcy|             Grocery|2026-05-12 12:30:...|dbfs:/Volumes/eco...|
|          toy|        Toys & Games|2026-05-12 12:30:...|dbfs:/Volumes/eco...|
|          spt|   Sports & Outdoors|2026-05-12 12:30:...|dbfs:/Volumes/eco...|
|          app|             Apparel|2026-05-12 12:30:...|dbfs:/Volumes/eco...|
|         grcy|             Grocery|2026-05-12 12:30

In [0]:
df_duplicate = df_bronze.groupBy("category_code").count().filter(f.col("count") > 1)
df_duplicate.show()

+-------------+-----+
|category_code|count|
+-------------+-----+
|          app|    2|
|         grcy|    2|
+-------------+-----+



In [0]:
df_silver = df_bronze.dropDuplicates(['category_code'])
display(df_silver)

category_code,category_name,_ingested_at,_source_file
ce,Electronics,2026-05-12T12:30:01.372Z,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv
app,Apparel,2026-05-12T12:30:01.372Z,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv
hnk,Home & Kitchen,2026-05-12T12:30:01.372Z,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv
bpc,Beauty & Personal Care,2026-05-12T12:30:01.372Z,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv
bks,Books,2026-05-12T12:30:01.372Z,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv
grcy,Grocery,2026-05-12T12:30:01.372Z,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv
toy,Toys & Games,2026-05-12T12:30:01.372Z,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv
spt,Sports & Outdoors,2026-05-12T12:30:01.372Z,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv


In [0]:
df_silver = df_silver.withColumn("category_code", f.upper(f.col("category_code")))
display(df_silver)

category_code,category_name,_ingested_at,_source_file
CE,Electronics,2026-05-12T12:30:01.372Z,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv
APP,Apparel,2026-05-12T12:30:01.372Z,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv
HNK,Home & Kitchen,2026-05-12T12:30:01.372Z,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv
BPC,Beauty & Personal Care,2026-05-12T12:30:01.372Z,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv
BKS,Books,2026-05-12T12:30:01.372Z,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv
GRCY,Grocery,2026-05-12T12:30:01.372Z,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv
TOY,Toys & Games,2026-05-12T12:30:01.372Z,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv
SPT,Sports & Outdoors,2026-05-12T12:30:01.372Z,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv


In [0]:
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_category")

## Products

In [0]:
# Read the raw data from the bronze table (ecommerce.bronze.brz_calendar)
df_bronze = spark.read.table(f"{catalog_name}.bronze.brz_products")

# Get row and column count
row_count, column_count = df_bronze.count(), len(df_bronze.columns)

# Print the results
print(f"Row count: {row_count}")
print(f"Column count: {column_count}")

Row count: 50000
Column count: 14


In [0]:
display(df_bronze.limit(5))

product_id,sku,category_code,brand_code,color,size,material,weight_grams,length_cm,width_cm,height_cm,rating_count,file_name,ingest_timestamp
2000000000015,STCR-HNK-00001,hnk,stcr,White,One-Size,Coton,305g,"22,2",17.1,6.3,0,dbfs:/Volumes/ecommerce/source_data/raw/products/products.csv,2026-05-12T12:32:15.153Z
2000000000022,HMNS-HNK-00002,hnk,hmns,Silver,One-Size,Steel,682g,"18,2",12.3,3.7,1,dbfs:/Volumes/ecommerce/source_data/raw/products/products.csv,2026-05-12T12:32:15.153Z
2000000000039,NOVW-CE-00003,ce,novw,Purple,One-Size,Wood,243g,"18,2",13.9,4.2,0,dbfs:/Volumes/ecommerce/source_data/raw/products/products.csv,2026-05-12T12:32:15.153Z
2000000000046,URTL-APP-00004,app,urtl,Silver,S,Ruber,225g,"17,6",4.6,5.8,50,dbfs:/Volumes/ecommerce/source_data/raw/products/products.csv,2026-05-12T12:32:15.153Z
2000000000053,GGRN-GRC-00005,grcy,ggrn,Silver,One-Size,Ruber,455g,"27,2",15.8,7.4,-4,dbfs:/Volumes/ecommerce/source_data/raw/products/products.csv,2026-05-12T12:32:15.153Z


In [0]:
df_bronze.select("weight_grams").show(5, truncate=False)

+------------+
|weight_grams|
+------------+
|305g        |
|682g        |
|243g        |
|225g        |
|455g        |
+------------+
only showing top 5 rows


In [0]:
# replace 'g' with ''
df_silver = df_bronze.withColumn(
    "weight_grams",
    f.regexp_replace(f.col("weight_grams"), "g", "").cast(IntegerType())
)
df_silver.select("weight_grams").show(5, truncate=False)

+------------+
|weight_grams|
+------------+
|305         |
|682         |
|243         |
|225         |
|455         |
+------------+
only showing top 5 rows


In [0]:
df_silver.select("length_cm").show(3)

+---------+
|length_cm|
+---------+
|     22,2|
|     18,2|
|     18,2|
+---------+
only showing top 3 rows


In [0]:
# replace , with .
df_silver = df_silver.withColumn(
    "length_cm",
    f.regexp_replace(f.col("length_cm"), ",", ".").cast(FloatType())
)
df_silver.select("length_cm").show(3)

+---------+
|length_cm|
+---------+
|     22.2|
|     18.2|
|     18.2|
+---------+
only showing top 3 rows


In [0]:
# convert category_code and brand_code to upper case
df_silver = df_silver.withColumn(
    "category_code",
    f.upper(f.col("category_code"))
).withColumn(
    "brand_code",
    f.upper(f.col("brand_code"))
)
df_silver.select("category_code", "brand_code").show(2)

+-------------+----------+
|category_code|brand_code|
+-------------+----------+
|          HNK|      STCR|
|          HNK|      HMNS|
+-------------+----------+
only showing top 2 rows


In [0]:
# Fix spelling mistakes
df_silver = df_silver.withColumn(
    "material",
    f.when(f.col("material") == "Coton", "Cotton")
     .when(f.col("material") == "Alumium", "Aluminum")
     .when(f.col("material") == "Ruber", "Rubber")
     .otherwise(f.col("material"))
)
df_silver.select("material").distinct().show()    

+---------+
| material|
+---------+
|   Cotton|
|    Steel|
|     Wood|
|   Rubber|
|  Plastic|
|Polyester|
|    Glass|
| Aluminum|
|    Paper|
|  Leather|
+---------+



In [0]:
df_silver.filter(f.col('rating_count')<0).select("rating_count").show(3)


+------------+
|rating_count|
+------------+
|          -4|
|          -2|
|          -2|
+------------+
only showing top 3 rows


In [0]:
# Convert negative rating_count to positive
df_silver = df_silver.withColumn(
    "rating_count",
    f.when(f.col("rating_count").isNotNull(), f.abs(f.col("rating_count")))
     .otherwise(f.lit(0))  # if null, replace with 0
)

In [0]:
# Check final cleaned data

df_silver.select(
    "weight_grams",
    "length_cm",
    "category_code",
    "brand_code",
    "material",
    "rating_count"
).show(10, truncate=False)

+------------+---------+-------------+----------+---------+------------+
|weight_grams|length_cm|category_code|brand_code|material |rating_count|
+------------+---------+-------------+----------+---------+------------+
|305         |22.2     |HNK          |STCR      |Cotton   |0           |
|682         |18.2     |HNK          |HMNS      |Steel    |1           |
|243         |18.2     |CE           |NOVW      |Wood     |0           |
|225         |17.6     |APP          |URTL      |Rubber   |50          |
|455         |27.2     |GRCY         |GGRN      |Rubber   |4           |
|232         |28.0     |BPC          |SLKE      |Plastic  |0           |
|507         |27.2     |CE           |VOLT      |Plastic  |5           |
|261         |27.7     |APP          |CBLT      |Polyester|0           |
|59          |12.5     |SPT          |ARFT      |Plastic  |11          |
|238         |10.7     |APP          |MOSA      |Polyester|6           |
+------------+---------+-------------+----------+--

In [0]:
# Write raw data to the silver layer (catalog: ecommerce, schema: silver, table: slv_dim_products)
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_products")